Question: Implement a simple Protocol Buffers server and client that defines a `Book` message with fields `title` (string), `author` (string), and `page_count` (integer). Include a service `Library` with a method `checkoutBook` that takes a `Book` message and returns the same `Book` message.


In [26]:
%%writefile ../schema/book.proto
syntax = "proto3";

message Book {
  string title = 1;
  string author = 2;
  int32 page_count = 3;
}

service Library {
  rpc checkoutBook(Book) returns (Book) {}
  }

Overwriting ../schema/book.proto


python -m grpc_tools.protoc -I./schema --python_out=. --grpc_python_out=. ./schema/book.proto
# run this in terminal

In [30]:
%%writefile ../book_protobuf_server.py
from concurrent import futures
import grpc
import os
import book_pb2
import book_pb2_grpc


class library(book_pb2_grpc.LibraryServicer):

  def checkoutBook(self, request, context):
    print(f"Received book checkout request: {request}")
    return request

server = grpc.server(futures.ThreadPoolExecutor(max_workers=2))
book_pb2_grpc.add_LibraryServicer_to_server(Library(), server)
server.add_insecure_port('[::]:50051')
print("Starting and running the server...")
print("Press Ctrl+C to stop the server.")
server.start()
server.wait_for_termination()

Overwriting ../book_protobuf_server.py


In [28]:
%%writefile ../book_client.py

import sys
sys.path.append('..')
import grpc
import book_pb2_grpc
import book_pb

Overwriting ../book_client.py


In [33]:
%%writefile ../book_client.py

import sys
sys.path.append('..')
import grpc
import book_pb2_grpc
import book_pb

def checkout_fn(stub, book1):
    book1 = stub.checkoutBook(book1)
    return book1

with grpc.insecure_channel('localhost:50051') as channel:
    stub = book_pb2_grpc.LibraryStub(channel)
    book_info = book_pb2.Book(title="Singapore History", author="Raffles", page_count=208)
    response = checkout_fn(stub, book_info)
    print("Checked out book:", response) 

Overwriting ../book_client.py
